# Алгоритмы кластеризации — практика
**Время: ~50–60 мин**

В этом блокноте разберём три классических алгоритма:
1. **K-Means** — центроидная кластеризация и метод локтя
2. **Agglomerative clustering** — иерархическая кластеризация и дендрограммы
3. **DBSCAN** — кластеризация по плотности и работа с шумом

Сначала поработаем с игрушечными данными (sklearn), затем применим всё к **эмбеддингам лиц** из датасета Nikolsky (InsightFace) и сравним результаты.

## 0. Импорты и игрушечные данные

Для экспериментов будем использовать `make_blobs` и `make_moons` из sklearn — данные с «очевидными» кластерами и без.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage
import json
from pathlib import Path

# Игрушечные данные: три компактных кластера
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=0.8, random_state=42)
X_blobs = StandardScaler().fit_transform(X_blobs)

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
ax.scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='viridis', alpha=0.8)
ax.set_title('Игрушечные данные: 3 кластера (истинные метки)')
plt.tight_layout()
plt.show()

---
## 1. K-Means

**Идея:** задаём число кластеров $K$. Алгоритм итеративно:
1. Назначает каждую точку ближайшему центроиду.
2. Пересчитывает центроиды как среднее точек в кластере.

Минимизируется сумма квадратов расстояний точек до центроидов (inertia). Недостаток: нужно заранее задать $K$.

In [ ]:
# K-Means на игрушечных данных (K=3)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
labels_kmeans = kmeans.fit_predict(X_blobs)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels_kmeans, cmap='viridis', alpha=0.8)
axes[0].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], 
               marker='*', s=300, c='red', edgecolors='black', linewidths=2, label='центроиды')
axes[0].set_title('K-Means (K=3)')
axes[0].legend()

axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_blobs, cmap='viridis', alpha=0.8)
axes[1].set_title('Истинные метки')
plt.tight_layout()
plt.show()

print('ARI (совпадение с истиной):', adjusted_rand_score(y_blobs, labels_kmeans).round(4))

### Метод локтя (Elbow)

Чтобы подобрать $K$ без знания истинных меток, считают **inertia** (сумму квадратов расстояний до центроидов) для разных $K$. На графике ищут «локоть» — после него прирост качества замедляется.

In [ ]:
K_range = range(2, 11)
inertias = []
silhouettes = []
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_blobs)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_blobs, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Метод локтя')
axes[0].axvline(3, color='gray', linestyle='--', alpha=0.7, label='K=3')
axes[0].legend()

axes[1].plot(K_range, silhouettes, 'go-')
axes[1].set_xlabel('K')
axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette (выше — лучше)')
axes[1].axvline(3, color='gray', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

**Упражнение 1 (K-Means).** Сгенерируйте данные с 4 кластерами (`make_blobs(centers=4, ...)`), постройте график локтя и silhouette для K от 2 до 8. Подберите K по локтю и по максимуму silhouette. Сравните с истинным K=4.

In [ ]:
# Ваш код здесь
# X4, y4 = make_blobs(...)
# ...

---
## 2. Agglomerative (иерархическая) кластеризация

**Идея:** начинаем с того, что каждая точка — свой кластер ($n$ кластеров). На каждом шаге объединяем два ближайших кластера, пока не получим нужное число кластеров или один кластер.

**Связь (linkage):** как измерять «расстояние» между кластерами:
- **single** — минимум расстояний между точками двух кластеров (цепочки)
- **complete** — максимум (компактные кластеры)
- **ward** — минимизация дисперсии внутри объединённого кластера (часто даёт сбалансированные кластеры)

In [ ]:
# Дендрограмма: по вертикали — расстояние, на котором два кластера объединились
linkage_matrix = linkage(X_blobs, method='ward')
plt.figure(figsize=(10, 5))
dendrogram(linkage_matrix, truncate_mode='lastp', p=30)
plt.title('Дендрограмма (Ward)')
plt.xlabel('Номер выборки / размер кластера')
plt.ylabel('Расстояние')
plt.tight_layout()
plt.show()

In [ ]:
# Agglomerative с n_clusters=3 (по дендрограмме видно разумный уровень отсечения)
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
labels_agg = agg.fit_predict(X_blobs)

plt.figure(figsize=(6, 5))
plt.scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels_agg, cmap='viridis', alpha=0.8)
plt.title('Agglomerative Clustering (ward, K=3)')
plt.tight_layout()
plt.show()
print('ARI:', adjusted_rand_score(y_blobs, labels_agg).round(4))

**Упражнение 2 (Agglomerative).** Постройте дендрограммы для тех же данных с `linkage='single'` и `linkage='complete'`. Сравните форму «дерева». Затем обучите Agglomerative с `n_clusters=3` для single и complete и сравните ARI с ward.

In [ ]:
# Ваш код здесь

---
## 3. DBSCAN

**Идея:** кластер — это плотная область. Параметры:
- **eps** — радиус окрестности;
- **min_samples** — минимум точек в окрестности, чтобы точка считалась «ядровой».

Точки делятся на: ядровые (в плотной области), граничные (попали в окрестность ядра) и **шум** (label = -1). Число кластеров задавать не нужно.

In [ ]:
# Данные с «лунами» — два изогнутых кластера (K-Means с ними справляется плохо)
X_moons, y_moons = make_moons(n_samples=300, noise=0.08, random_state=42)
X_moons = StandardScaler().fit_transform(X_moons)

db = DBSCAN(eps=0.35, min_samples=5)
labels_db = db.fit_predict(X_moons)

n_clusters = len(set(labels_db) - {-1})
n_noise = (labels_db == -1).sum()
print(f'Кластеров: {n_clusters}, шумовых точек: {n_noise}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=labels_db, cmap='viridis', alpha=0.8)
axes[0].set_title('DBSCAN')
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis', alpha=0.8)
axes[1].set_title('Истинные метки')
plt.tight_layout()
plt.show()

In [ ]:
# Для сравнения: K-Means на лунах (ожидаемо хуже)
km_moons = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_km_moons = km_moons.fit_predict(X_moons)
plt.figure(figsize=(6, 5))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=labels_km_moons, cmap='viridis', alpha=0.8)
plt.title('K-Means на лунах (K=2)')
plt.show()
print('ARI DBSCAN:', adjusted_rand_score(y_moons, labels_db).round(4))
print('ARI K-Means:', adjusted_rand_score(y_moons, labels_km_moons).round(4))

**Упражнение 3 (DBSCAN).** Добавьте к данным из `make_moons` случайные выбросы: 20 точек из `np.random.uniform(-2, 2, (20, 2))`. Запустите DBSCAN с разными `eps` (например 0.2, 0.35, 0.5) и посмотрите, как меняется число кластеров и доля шума. Подберите eps так, чтобы два «полумесяца» остались двумя кластерами, а выбросы ушли в шум.

In [ ]:
# Ваш код здесь

---
## 4. Эмбеддинги лиц: датасет Nikolsky и сравнение алгоритмов

Берём эмбеддинги лиц из датасета **NikolskyDataset_v1**: либо загружаем готовые из `results_clustering/face_clustering_metadata.json`, либо считаем их по фотографиям из папки с датасетом (`NikolskyDataset_v1` или `FaceTechSprintRudnNikolskyDataset_v1`) с помощью **InsightFace**. Путь к фото может быть локальным или Kaggle (`/kaggle/input/...`); эмбеддинги в JSON уже посчитаны и подходят для кластеризации. Затем кластеризуем эмбеддинги K-Means, Agglomerative и DBSCAN и сравниваем результаты (визуализация в 2D через PCA, метрики, размеры кластеров).

### 4.1 Загрузка эмбеддингов

Если есть файл `results_clustering/face_clustering_metadata.json` — читаем эмбеддинги оттуда. Иначе — ищем папку с фото (например `NikolskyDataset_v1`) и считаем эмбеддинги через InsightFace.

In [ ]:
def load_embeddings_from_json(json_path):
    """Загружает эмбеддинги и метаданные из face_clustering_metadata.json."""
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    embeddings = np.array([item['embedding'] for item in data])
    meta = [{'photo_name': item['photo_name'], 'face_index': item.get('face_index', 0)} for item in data]
    return embeddings, meta

json_path = Path('results_clustering/face_clustering_metadata.json')
if json_path.exists():
    X_faces, face_meta = load_embeddings_from_json(json_path)
    print(f'Загружено эмбеддингов: {len(X_faces)}, размерность: {X_faces.shape[1]}')
else:
    X_faces, face_meta = None, None
    print('Файл с эмбеддингами не найден. Будем считать эмбеддинги из папки с фото (см. следующую ячейку).')

In [ ]:
# Если эмбеддингов нет в JSON — считаем из папки NikolskyDataset_v1 через InsightFace
def compute_embeddings_from_folder(folder_path, max_photos=None):
    import cv2
    from insightface.app import FaceAnalysis
    folder = Path(folder_path)
    if not folder.exists():
        return None, None
    app = FaceAnalysis(providers=['CPUExecutionProvider'])
    app.prepare(ctx_id=0, det_size=(640, 640))
    images = sorted(list(folder.glob('*.jpg')) + list(folder.glob('*.png')))[:max_photos]
    all_embeddings = []
    meta = []
    for path in images:
        img = cv2.imread(str(path))
        if img is None:
            continue
        faces = app.get(img)
        for i, face in enumerate(faces):
            all_embeddings.append(face.embedding)
            meta.append({'photo_name': path.stem, 'face_index': i})
    return np.array(all_embeddings), meta

if X_faces is None:
    for name in ['NikolskyDataset_v1', 'FaceTechSprintRudnNikolskyDataset_v1', '.']:
        X_faces, face_meta = compute_embeddings_from_folder(name, max_photos=100)
        if X_faces is not None and len(X_faces) > 0:
            print(f'Посчитано эмбеддингов: {len(X_faces)} из папки {name}')
            break
    if X_faces is None:
        print('Папка с фото не найдена. Используйте results_clustering/face_clustering_metadata.json или положите фото в NikolskyDataset_v1.')

### 4.2 Кластеризация лиц и визуализация в 2D (PCA)

Эмбеддинги имеют размерность 512. Для картинки снижаем до 2 компонент с помощью PCA.

In [ ]:
if X_faces is not None and len(X_faces) > 10:
    from sklearn.decomposition import PCA
    X_faces_scaled = StandardScaler().fit_transform(X_faces)
    pca = PCA(n_components=2, random_state=42)
    X_faces_2d = pca.fit_transform(X_faces_scaled)
    print('Доля объяснённой дисперсии (2 компоненты):', pca.explained_variance_ratio_.round(4))

    # K-Means, Agglomerative, DBSCAN
    n_face = min(15, max(3, len(X_faces) // 20))  # примерное число людей/кластеров
    km_f = KMeans(n_clusters=n_face, random_state=42, n_init=10)
    agg_f = AgglomerativeClustering(n_clusters=n_face, linkage='ward')
    db_f = DBSCAN(eps=0.5, min_samples=3)

    labels_km_f = km_f.fit_predict(X_faces_scaled)
    labels_agg_f = agg_f.fit_predict(X_faces_scaled)
    labels_db_f = db_f.fit_predict(X_faces_scaled)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].scatter(X_faces_2d[:, 0], X_faces_2d[:, 1], c=labels_km_f, cmap='tab20', alpha=0.7)
    axes[0].set_title('K-Means')
    axes[1].scatter(X_faces_2d[:, 0], X_faces_2d[:, 1], c=labels_agg_f, cmap='tab20', alpha=0.7)
    axes[1].set_title('Agglomerative')
    axes[2].scatter(X_faces_2d[:, 0], X_faces_2d[:, 1], c=labels_db_f, cmap='tab20', alpha=0.7)
    axes[2].set_title('DBSCAN (шум = -1)')
    plt.suptitle('Кластеризация эмбеддингов лиц (2D PCA)')
    plt.tight_layout()
    plt.show()
else:
    print('Недостаточно эмбеддингов для кластеризации. Загрузите face_clustering_metadata.json или фото.')

In [ ]:
# Сводка: число кластеров и размеры
if X_faces is not None and len(X_faces) > 10:
    def cluster_stats(labels, name):
        unique, counts = np.unique(labels, return_counts=True)
        n_clusters = len(unique) if -1 not in unique else len(unique) - 1
        noise = (labels == -1).sum() if -1 in unique else 0
        print(f'{name}: кластеров {n_clusters}, шум {noise}, размеры кластеров: min={counts[counts > 0].min() if (labels != -1).any() else 0}, max={counts.max()}')
    cluster_stats(labels_km_f, 'K-Means')
    cluster_stats(labels_agg_f, 'Agglomerative')
    cluster_stats(labels_db_f, 'DBSCAN')
    print('\nSilhouette (чем выше, тем лучше):')
    print('  K-Means:', silhouette_score(X_faces_scaled, labels_km_f).round(4))
    print('  Agglomerative:', silhouette_score(X_faces_scaled, labels_agg_f).round(4))
    if len(set(labels_db_f) - {-1}) > 1:
        mask = labels_db_f != -1
        print('  DBSCAN (без шума):', silhouette_score(X_faces_scaled[mask], labels_db_f[mask]).round(4))

**Упражнение 4 (лица).** Подберите для эмбеддингов лиц:
1. Оптимальное K для K-Means по методу локтя (или silhouette).
2. Параметры DBSCAN (eps, min_samples), при которых получается разумное число кластеров без избыточного шума. Сравните визуально разметку в 2D (PCA) и размеры кластеров.

In [ ]:
# Ваш код: elbow/silhouette для K-Means на X_faces_scaled, подбор eps для DBSCAN